# Real-Time Fraud Detection: Model Training Pipeline

This notebook implements the complete data science pipeline for loading transaction data, performing exploratory analysis, engineering features, training a class-weighted XGBoost classifier, and exporting the serialized artifacts for production use.

### 1. Data Ingestion & Target Class Distribution

Load the credit card transactions dataset, check the class balance, and calculate the proportion of fraudulent transactions in the dataset.

In [2]:
import kagglehub

# Re-initialize and grab the cached path from kagglehub
path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")
print(f"Dataset path verified at: {path}")

/home/yarrmani/real_time_fraud_detection/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset path verified at: /home/yarrmani/.cache/kagglehub/datasets/mlg-ulb/creditcardfraud/versions/3


In [3]:
import os
import glob
import pandas as pd

dataset_dir = path 

csv_files = glob.glob(os.path.join(dataset_dir, "*.csv"))

if not csv_files:
    raise FileNotFoundError(f"No CSV files found in the dataset directory: {dataset_dir}")

target_csv = csv_files[0]
print(f"Loading data directly from production cache: {target_csv}")

df_real = pd.read_csv(target_csv)

print(f"\n[INFO] Dataset successfully loaded.")
print(f"Total Transactions: {df_real.shape[0]:,}")
print(f"Total Features:     {df_real.shape[1]}")

class_counts = df_real['Class'].value_counts()
fraud_percentage = df_real['Class'].value_counts(normalize=True) * 100

print("\n--- Target Variable Distribution ---")
print(f"Legitimate (Class 0): {class_counts[0]:,} ({fraud_percentage[0]:.3f}%)")
print(f"Fraudulent (Class 1): {class_counts[1]:,} ({fraud_percentage[1]:.3f}%)")

Loading data directly from production cache: /home/yarrmani/.cache/kagglehub/datasets/mlg-ulb/creditcardfraud/versions/3/creditcard.csv

[INFO] Dataset successfully loaded.
Total Transactions: 284,807
Total Features:     31

--- Target Variable Distribution ---
Legitimate (Class 0): 284,315 (99.827%)
Fraudulent (Class 1): 492 (0.173%)


### 2. Exploratory Data Analysis

Examine missing values across columns and compute summary statistics of key financial features (Time and Amount) segmented by class.

In [4]:
import numpy as np

missing_values = df_real.isnull().sum().sum()
print(f"Total missing values in dataset: {missing_values}")

print("\n=== Financial Feature Statistics ===")
print(df_real[['Time', 'Amount']].describe())

print("\n=== Transaction Amount Breakdown by Class ===")
print(df_real.groupby('Class')['Amount'].agg(['mean', 'median', 'max', 'count']))


Total missing values in dataset: 0

=== Financial Feature Statistics ===
                Time         Amount
count  284807.000000  284807.000000
mean    94813.859575      88.349619
std     47488.145955     250.120109
min         0.000000       0.000000
25%     54201.500000       5.600000
50%     84692.000000      22.000000
75%    139320.500000      77.165000
max    172792.000000   25691.160000

=== Transaction Amount Breakdown by Class ===
             mean  median       max   count
Class                                      
0       88.291022   22.00  25691.16  284315
1      122.211321    9.25   2125.87     492


### 3. Feature Engineering, Model Training, and Evaluation

Transform time to hour-of-day, scale transaction amounts using a RobustScaler to handle extreme outliers, partition the dataset into stratified training and testing sets, compute positive class weight parameters, train an XGBoost classifier, evaluate it using a Precision-Recall classification report and confusion matrix, and serialize the trained model and scaler to the artifacts directory.

In [5]:
import os
import joblib
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.preprocessing import RobustScaler

df_model = df_real.copy()

df_model['Hour'] = (df_model['Time'] / 3600).astype(int) % 24

scaler = RobustScaler()
df_model['Amount_Scaled'] = scaler.fit_transform(df_model[['Amount']])

df_model = df_model.drop(columns=['Time', 'Amount'])

print(f"Final shape: {df_model.shape}")
print(f"Columns: {list(df_model.columns)}")

X = df_model.drop(columns=['Class'])
y = df_model['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"\nTrain shape: {X_train.shape} | Fraud cases: {y_train.sum()}")
print(f"Test shape:  {X_test.shape}  | Fraud cases: {y_test.sum()}")

scale_pos_weight = (y_train == 0).sum() / y_train.sum()
print(f"scale_pos_weight: {scale_pos_weight:.2f}")

print("\nTraining production XGBoost Classifier...")
model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=scale_pos_weight,
    eval_metric='aucpr',
    random_state=42,
    n_jobs=-1,
    subsample=0.8,
    colsample_bytree=0.8
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=50
)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("\n=== Classification Report ===")
print(classification_report(y_test, y_pred, target_names=['Legitimate', 'Fraud']))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob):.4f}")

print("\n=== Confusion Matrix ===")
cm = confusion_matrix(y_test, y_pred)
print(cm)
print(f"\nTrue Positives (caught fraud):    {cm[1][1]}")
print(f"False Negatives (missed fraud):   {cm[1][0]}")
print(f"False Positives (false alarms):   {cm[0][1]}")
print(f"True Negatives (correct legit):   {cm[0][0]}")

print("\n=== Top 10 Most Important Features ===")
importances = pd.Series(
    model.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)
print(importances.head(10))

os.makedirs('model_artifacts', exist_ok=True)
joblib.dump(model, 'model_artifacts/fraud_model.pkl')
joblib.dump(scaler, 'model_artifacts/scaler.pkl')

feature_columns = list(X_train.columns)
joblib.dump(feature_columns, 'model_artifacts/feature_columns.pkl')

print("\n=== Export Status ===")
print("Saved artifacts to 'model_artifacts/' directory successfully.")

Final shape: (284807, 31)
Columns: ['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Class', 'Hour', 'Amount_Scaled']

Train shape: (227845, 30) | Fraud cases: 394
Test shape:  (56962, 30)  | Fraud cases: 98
scale_pos_weight: 577.29

Training production XGBoost Classifier...
[0]	validation_0-aucpr:0.48268
[50]	validation_0-aucpr:0.69913
[100]	validation_0-aucpr:0.80984
[150]	validation_0-aucpr:0.84862
[200]	validation_0-aucpr:0.86824
[250]	validation_0-aucpr:0.87486
[299]	validation_0-aucpr:0.87676

=== Classification Report ===
              precision    recall  f1-score   support

  Legitimate       1.00      1.00      1.00     56864
       Fraud       0.86      0.84      0.85        98

    accuracy                           1.00     56962
   macro avg       0.93      0.92      0.92     56962
weighted avg       1.00      1.00      1.00     56962


### 4. Pipeline Verification and Smoke Testing

Reload the serialized classifier, scaling parameters, and feature column schemas. Perform a dummy prediction to confirm end-to-end functionality of the deployment pipeline.

In [7]:
import joblib
import numpy as np

model_check   = joblib.load('model_artifacts/fraud_model.pkl')
scaler_check  = joblib.load('model_artifacts/scaler.pkl')
features_check = joblib.load('model_artifacts/feature_columns.pkl')

print(f"Features loaded: {len(features_check)}")
print(f"Feature list: {features_check}")

dummy = np.zeros((1, len(features_check)))
prob = model_check.predict_proba(dummy)[0][1]
print(f"\nDummy fraud probability: {prob:.4f}")
print("Model loads and predicts correctly." if prob is not None else "PROBLEM")

Features loaded: 30
Feature list: ['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Hour', 'Amount_Scaled']

Dummy fraud probability: 0.0002
Model loads and predicts correctly.
